In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q transformers accelerate bitsandbytes langchain pydantic sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

base_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)

In [ ]:
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb_cfg, device_map="auto")

save_dir = "Qwen/Qwen2.5-7B-Instruct_4bit_bnb"
try:
    model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    print("Saved quantized model to:", save_dir)
except Exception as e:
    print("Save 4-bit not supported in this env:", e)

In [ ]:
import time
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

save_dir = "/kaggle/working/Qwen/Qwen2.5-7B-Instruct_4bit_bnb"

def load_model_and_tokenizer(path_or_id):
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(path_or_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        path_or_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    return tokenizer, model

In [ ]:
if Path(save_dir).exists():
    print(f"Loading local 4-bit checkpoint: {save_dir}")
    tok, model = load_model_and_tokenizer(save_dir)
else:
    print("Local 4-bit checkpoint not found (saving likely not supported in this env).")

In [ ]:
def generate_text(prompt, max_new_tokens=2048, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1] # new
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    # generated_texts = []
    # for output in outputs:
    #     generated_tokens = output[input_length:]  # Skip input tokens
    #     decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    #     generated_texts.append(decoded)

In [ ]:
from langchain.llms.base import LLM
from typing import Any

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt, max_new_tokens=2048)[0]
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [12]:
from langchain import LLMChain, PromptTemplate
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# Define response schemas to get strict format instructions (top-level only)
response_schemas = [
    ResponseSchema(
        name="technical_questions",
        description="Array of objects: {question, suggested_answer, confidence_score (0-1)}"
    ),
    ResponseSchema(
        name="behavioral_questions",
        description="Array of objects: {question, suggested_answer, confidence_score (0-1)}"
    )
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()
print("Format instructions:\n", format_instructions)


Format instructions:
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"technical_questions": string  // Array of objects: {question, suggested_answer, confidence_score (0-1)}
	"behavioral_questions": string  // Array of objects: {question, suggested_answer, confidence_score (0-1)}
}
```


In [13]:
# %% [code]
prompt_template = PromptTemplate(
    input_variables=["resume", "job_description", "format_instructions"],
    template="""
You are an expert interview coach and technical hiring specialist.

Given the candidate resume:
---
{resume}
---

And the job description:
---
{job_description}
---

Using evidence-based best practices, produce a mock interview tailored to this candidate and job.

Requirements:
- Produce both technical and behavioral questions.
- For each question include: suggested_answer and confidence_score (0.0 - 1.0).
- Confidence is how well the candidate's resume maps to the skills/experience needed to answer the question.
- Use the job description to prioritize topics; prioritize role-specific technical questions.
- Keep total technical questions ≈ 8-12 and behavioral ≈ 6-8 (but model's judgment is fine).
- Do NOT output any extra prose, only produce the JSON in the structure described below.

{format_instructions}
"""
)


In [17]:
from pydantic import BaseModel, ValidationError
from typing import Any, List, Dict, Optional

class QuestionItem(BaseModel):
    question: str
    suggested_answer: str
    confidence_score: Optional[float] = None

class InterviewOutput(BaseModel):
    technical_questions: List[QuestionItem]
    behavioral_questions: List[QuestionItem]

In [18]:
# %% [code]
def extract_first_json(text: str) -> str:
    """
    Find the first { ... } JSON-like substring in text.
    Simple and robust: find first '{' and last '}' after it.
    """
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object found.")
    # find matching closing brace by scanning (keeps nested braces safe-ish)
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    # fallback
    raise ValueError("Could not extract balanced JSON object.")

# Simple token overlap confidence fallback
STOPWORDS = {
    "the","and","a","an","of","to","for","in","on","with","is","are","by","as","that","this",
    "be","will","their","at","from","it","or","we","you","your","i","my","our"
}

def tokenize(text: str):
    tokens = re.findall(r"[A-Za-z0-9\+\#\-\_\.]+", text.lower())
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return set(tokens)

def compute_confidence_by_overlap(resume_text: str, jd_text: str, question_text: str) -> float:
    """
    Score how well the resume covers skills/concepts related to question.
    Approach: token overlap between (resume tokens) and (jd tokens ∪ question tokens).
    Returns float in [0,1].
    """
    r_tokens = tokenize(resume_text)
    jd_tokens = tokenize(jd_text)
    q_tokens = tokenize(question_text)
    target = jd_tokens.union(q_tokens)
    if not target:
        return 0.0
    overlap = r_tokens.intersection(target)
    score = len(overlap) / max(1, len(target))
    # map to [0,1], small boost
    return min(1.0, score * 1.15)


In [19]:
# %% [code]
# Example inputs (replace with file-parsed content)
resume_text = """
John Doe
Software Engineer with 3 years experience building backend systems.
Experience:
- Built REST APIs using Python (Django, FastAPI), containerized apps with Docker.
- Deployed ML models using PyTorch to production (simple model serving infra).
- Worked with SQL databases, wrote unit & integration tests.
Skills: Python, FastAPI, Django, PyTorch, SQL, Docker, CI/CD
"""

job_desc = """
Backend Engineer (Python)
- Build and maintain production Python APIs and microservices.
- Deploy and maintain ML inference pipelines.
- Ensure code quality with tests, CI/CD, and code reviews.
- Work cross-functionally with Data and Frontend teams.
"""

# create chain and run
chain = LLMChain(llm=llm, prompt=prompt_template)

raw = chain.run(resume=resume_text, job_description=job_desc, format_instructions=format_instructions)
print("---- RAW MODEL OUTPUT ----\n")
print(raw)  


/tmp/ipykernel_47/2337290132.py:22: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
/tmp/ipykernel_47/2337290132.py:24: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw = chain.run(resume=resume_text, job_description=job_desc, format_instructions=format_instructions)


---- RAW MODEL OUTPUT ----


You are an expert interview coach and technical hiring specialist.

Given the candidate resume:
---

John Doe
Software Engineer with 3 years experience building backend systems.
Experience:
- Built REST APIs using Python (Django, FastAPI), containerized apps with Docker.
- Deployed ML models using PyTorch to production (simple model serving infra).
- Worked with SQL databases, wrote unit & integration tests.
Skills: Python, FastAPI, Django, PyTorch, SQL, Docker, CI/CD

---

And the job description:
---

Backend Engineer (Python)
- Build and maintain production Python APIs and microservices.
- Deploy and maintain ML inference pipelines.
- Ensure code quality with tests, CI/CD, and code reviews.
- Work cross-functionally with Data and Frontend teams.

---

Using evidence-based best practices, produce a mock interview tailored to this candidate and job.

Requirements:
- Produce both technical and behavioral questions.
- For each question include: suggested_ans

In [23]:
import json
import re

def extract_clean_json(text):
    fenced = re.findall(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if fenced:
        return fenced[-1]
    loose = re.findall(r"(\{(?:.|\n)*?\})", text)
    if loose:
        return loose[-1]
    raise ValueError("No JSON found in text.")

# Extract JSON
json_str = extract_clean_json(raw)

# Parse JSON
parsed = json.loads(json_str)

def normalize_list(obj):
    if obj is None:
        return []
    if isinstance(obj, str):
        try:
            return json.loads(obj)
        except:
            return []
    if isinstance(obj, list):
        return obj
    return []

tech = normalize_list(parsed.get("technical_questions"))
beh = normalize_list(parsed.get("behavioral_questions"))

def fill_questions(q_list, resume_text, job_desc):
    out = []
    for q in q_list:
        q_obj = {}
        q_obj["question"] = q.get("question") or q.get("q") or str(q)
        q_obj["suggested_answer"] = (
            q.get("suggested_answer")
            or q.get("answer")
            or q.get("suggestion")  # fix for model variation
            or ""
        )
        conf = q.get("confidence_score")
        if conf is None:
            conf = compute_confidence_by_overlap(resume_text, job_desc, q_obj["question"])
        else:
            try:
                conf = float(conf)
                conf = max(0.0, min(1.0, conf))
            except:
                conf = compute_confidence_by_overlap(resume_text, job_desc, q_obj["question"])
        q_obj["confidence_score"] = conf

        out.append(QuestionItem(**q_obj))

    return out

tech_items = fill_questions(tech, resume_text, job_desc)
beh_items = fill_questions(beh, resume_text, job_desc)

interview_output = InterviewOutput(
    technical_questions=tech_items,
    behavioral_questions=beh_items
)

print(interview_output.model_dump_json(indent=2))


{
  "technical_questions": [
    {
      "question": "Can you explain the difference between Django and FastAPI?",
      "suggested_answer": "Django is a high-level Python web framework that encourages rapid development and clean, pragmatic design. It includes an ORM, admin panel, and other built-in features. FastAPI is a modern, fast (high-performance) web framework for building APIs with Python 3.7+ based on standard Python type hints. It also generates interactive API documentation similar to Swagger.",
      "confidence_score": 0.9
    },
    {
      "question": "What is Docker and how do you use it to containerize your applications?",
      "suggested_answer": "Docker is an open-source platform for developing, deploying, and running applications inside containers. To containerize an application, you create a Dockerfile with instructions for setting up the environment and dependencies, then build the image and run the container. For example, you might write a Dockerfile specifying 